In [1]:
import pandas as pd
import numpy as np
import torch
from matplotlib import pyplot as plt
from scipy import sparse as sps
from tqdm import tqdm
from sklearn.metrics import ndcg_score
# from elsa import ELSA

%matplotlib inline

In [2]:
df_part1 = pd.read_csv('transaction_data_part1.csv')
df_part2 = pd.read_csv('transaction_data_part2.csv')
df = pd.concat([df_part1, df_part2], ignore_index=True)
df.columns = df.columns.str.lower()
df.head()

,household_key,basket_id,day,product_id,quantity,sales_value,store_id,retail_disc,trans_time,week_no,coupon_disc,coupon_match_disc
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0


In [3]:
print("Unique household keys", len(df.household_key.unique())) # N
print("Unique products", len(df.product_id.unique())) # M
print("Unique baskets", len(df.basket_id.unique()))

Unique household keys 2500
Unique products 92339
Unique baskets 276484


In [4]:
last_day = max(df.day)
n_users = 10000

In [5]:
already_data = pd.read_csv("current.csv")
already_data.head(5)

,basket_id,train_interactions,ease_preds
0,27021090316,"[874061, 909349, 939636, 945200, 960110, 98176...","[np.int64(1082185), np.int64(1068719), np.int6..."
1,27021162328,"[904181, 992208, 1007764, 1011918, 6442448, 80...","[np.int64(874061), np.int64(820347), np.int64(..."
2,27031090413,"[1036081, 1079174, 1079482]","[np.int64(909396), np.int64(9553183), np.int64..."
3,27031233213,"[842264, 842470, 844179, 849129, 861419, 88170...","[np.int64(1082185), np.int64(826249), np.int64..."
4,27057645831,[6514134],"[np.int64(874061), np.int64(820347), np.int64(..."


In [6]:
# all_baskets = np.random.choice(np.unique(df.basket_id), size=n_users, replace=False)
all_baskets = already_data['basket_id'].to_numpy()
print(all_baskets.shape)
# extra_baskets = np.random.choice(np.unique(df.basket_id), size=n_users - all_baskets.shape[0], replace=False)

(8640,)


In [7]:
test_thr = 30
valid_thr = 90

train_df = df.loc[(df.day < last_day - valid_thr)].copy()
val_df = df.loc[(last_day - valid_thr <= df.day) & (df.day < last_day - test_thr)].copy()
test_df = df.loc[(last_day - test_thr <= df.day)].copy()

train_df = train_df.loc[train_df.basket_id.isin(all_baskets)].copy()
# val_df = val_df.loc[val_df.basket_id.isin(all_baskets)].copy()
# test_df = test_df.loc[test_df.basket_id.isin(all_baskets)].copy()

In [8]:
print("Shapes:", train_df.shape, val_df.shape, test_df.shape)

Shapes: (78900, 12) (249586, 12) (130448, 12)


In [9]:
def filter_column(df, col, min_freq):
    cnt = df[col].value_counts()
    ok_values = cnt[cnt >= min_freq].index
    return df[df[col].isin(ok_values)].copy()

def filter_dataframe(df, cols, min_freq=5):
    df_copy = df.copy()
    prev = (0, 0)
    cur = df_copy.shape
    while (prev != cur):
        prev = cur
        for col in cols:
            df_copy = filter_column(df_copy, col, min_freq)
        cur = df_copy.shape
    return df_copy

# filtered_train = train_df
filtered_train = filter_dataframe(train_df, ['product_id'], min_freq=3)

In [10]:
print("Shape of filtered train_df", filtered_train.shape)

Shape of filtered train_df (60337, 12)


In [11]:
def df_encode(df):
    product2id = {k: v for v, k in enumerate(df.product_id.unique())}
    basket2id = {k: v for v, k in enumerate(df.basket_id.unique())}
    id2product = {v: k for k, v in product2id.items()}
    id2basket = {v: k for k, v in basket2id.items()}
    df1 = df.copy()
    df1['basket_id'] = df1.basket_id.map(basket2id)
    df1['product_id'] = df1.product_id.map(product2id)
    return df1, product2id, basket2id, id2product, id2basket

filtered_train, product2id_train, basket2id_train, id2product_train, id2basket_train = df_encode(filtered_train)
filtered_train.sample(5)

,household_key,basket_id,day,product_id,quantity,sales_value,store_id,retail_disc,trans_time,week_no,coupon_disc,coupon_match_disc
171710,1660,622,101,1840,3,2.64,381,-1.53,1113,15,0.0,0.0
1007921,1968,3696,318,453,1,2.49,321,0.00,1749,46,0.0,0.0
344598,1122,1279,149,697,1,0.88,372,-0.51,1848,22,0.0,0.0
416707,2151,1545,168,4473,2,0.60,366,-0.59,1221,25,0.0,0.0
1936077,552,6994,550,6515,1,5.34,298,-1.23,1253,79,0.0,0.0


In [12]:
print(max(filtered_train.basket_id), max(filtered_train.product_id))

7959 7092


In [13]:
# device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [14]:
def fit_ease(X, reg=100):
    if sps.issparse(X):
        X = X.toarray()
    X_t = torch.tensor(X, dtype=torch.float32, device=device)
    n_items = X_t.size(1)
    G = X_t.T @ X_t + reg * torch.eye(n_items, device=device)
    P = torch.linalg.inv(G)
    diagP = torch.diag(P)
    B = P / (-diagP.unsqueeze(0))
    B.fill_diagonal_(0)
    return B.cpu().numpy()

matrix = sps.coo_matrix(
    (np.ones(filtered_train.shape[0]), (filtered_train['basket_id'], filtered_train['product_id'])),
    shape=(filtered_train['basket_id'].max() + 1, filtered_train['product_id'].max() + 1),
)

W = fit_ease(matrix)

In [15]:
train_grouped = train_df.groupby('basket_id').apply(
    lambda x: [t1 for t1 in sorted(x.product_id)]
).reset_index()

train_grouped.rename({0:'train_interactions'}, axis=1, inplace=True)
train_grouped.head()

,basket_id,train_interactions
0,27021090316,"[874061, 909349, 939636, 945200, 960110, 98176..."
1,27021162328,"[904181, 992208, 1007764, 1011918, 6442448, 80..."
2,27031090413,"[1036081, 1079174, 1079482]"
3,27031233213,"[842264, 842470, 844179, 849129, 861419, 88170..."
4,27057645831,[6514134]


In [16]:
def get_preds_batch(joined, W, item2id_, id2item_, topn=10):
    n_users = len(joined)
    n_items = W.shape[0]

    rows = []
    cols = []
    user_seen = []

    for idx, (_, row) in enumerate(joined.iterrows()):
        seen = []
        for item in row['train_interactions']:
            if item in item2id_:
                encoded = item2id_[item]
                rows.append(idx)
                cols.append(encoded)
                seen.append(encoded)
        user_seen.append(seen)

    rows = np.array(rows)
    user_matrix = sps.csr_matrix(
        (np.ones(len(rows)), (rows, cols)),
        shape=(n_users, n_items)
    )

    # print(W.shape, len(rows), len(cols))
    all_scores = user_matrix @ W

    all_preds = []
    for idx in range(n_users):
        scores = np.array(all_scores[idx].copy()).ravel()
        scores[user_seen[idx]] = -1e9
        top_idx = np.argpartition(scores, -topn)[-topn:]
        top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
        all_preds.append([id2item_[i] for i in top_idx if i in id2item_])

    return all_preds
train_grouped['ease_preds'] = get_preds_batch(train_grouped, W, product2id_train, id2product_train, topn=10)
train_grouped.sample(10)

,basket_id,train_interactions,ease_preds
4474,32137589066,[6534178],"[1037840, 1075368, 1006483, 1036939, 1043920, ..."
279,27785722316,"[820801, 836543, 910032, 935463, 988791, 10364...","[1082185, 5569230, 1098066, 860776, 951590, 11..."
6242,33906161167,"[821083, 838186, 839094, 862349, 906763, 91286...","[1127831, 908531, 995242, 1029743, 995785, 110..."
3480,31198887480,"[901656, 938190, 939476, 947728, 979733, 98141...","[995242, 1029743, 1127831, 1106523, 1023720, 1..."
4284,31981291442,"[997096, 1023761]","[9526563, 1010079, 1042616, 5566809, 861494, 1..."
2333,30036995512,"[834427, 1035207, 6773196]","[1034686, 896974, 1106523, 1112473, 937571, 95..."
7773,40248387515,"[938863, 1121393, 16766021]","[995785, 870515, 841220, 1050229, 1082185, 844..."
2247,30033296824,"[934097, 9527158]","[7440665, 7440663, 995242, 861701, 844179, 913..."
5122,32823646518,"[879755, 1107553]","[1085604, 5978656, 1037894, 916990, 1073150, 1..."
1450,28941895793,"[826249, 837107, 861675, 868645, 868953, 87387...","[1098066, 1029743, 1082185, 1106523, 995242, 1..."


In [17]:
products = pd.read_csv('product.csv')
products.columns = products.columns.str.lower()
products.head()

,product_id,manufacturer,department,brand,commodity_desc,sub_commodity_desc,curr_size_of_product
0,25671,2,GROCERY,National,FRZN ICE,ICE - CRUSHED/CUBED,22 LB
1,26081,2,MISC. TRANS.,National,NO COMMODITY DESCRIPTION,NO SUBCOMMODITY DESCRIPTION,
2,26093,69,PASTRY,Private,BREAD,BREAD:ITALIAN/FRENCH,
3,26190,69,GROCERY,Private,FRUIT - SHELF STABLE,APPLE SAUCE,50 OZ
4,26355,69,GROCERY,Private,COOKIES/CONES,SPECIALTY COOKIES,14 OZ


In [18]:
product_id_to_title = products.set_index('product_id')['sub_commodity_desc'].to_dict()
product_id_to_title_main = products.set_index('product_id')['commodity_desc'].to_dict()
product_id_to_title_weight = products.set_index('product_id')['curr_size_of_product'].to_dict()

**ФОРМУЛЫ ЧТОБЫ НЕ СМОТРЕТЬ ПОСТОЯННО В СТАТЬЮ**

1. Вычислить семантическую матрицу корреляции товаров $S$ из LLM-эмбеддингов $F$:
$$P_F = (F^\top F + \lambda_F I)^{-1}, \quad S = I - P_F \cdot \text{diagMat}(\mathbf{1} \oslash \text{diag}(P_F))$$

2. Обучить матрицу весов $B$ на взаимодействиях, регуляризованную через $S$:
$$P_{KD} = (X^\top X + (\lambda_{KD} + \lambda_X)I)^{-1}$$
$$\mu = \text{diag}(I + \lambda_{KD} P_{KD} S) \oslash \text{diag}(P_{KD})$$
$$B_{L3AE} = I + \lambda_{KD} P_{KD} S - P_{KD} \cdot \text{diagMat}(\mu)$$

In [19]:
import torch
import torch as nn

In [20]:
%pip install sentence-transformers==2.7.0 transformers==4.38.0

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [21]:
%pip install torch torchvision torchaudio

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [22]:
from sentence_transformers import SentenceTransformer
# import sentence_transformers
import numpy as np
import scipy.sparse as sps

sbert_model = SentenceTransformer('all-mpnet-base-v2')
print("Размерность эмбеддинга:", sbert_model.get_sentence_embedding_dimension())

/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/home/jupyter/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/jupyter/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Размерность эмбеддинга: 768


In [23]:
# Описания
item_ids = list(product2id_train.keys())

item_texts = []
for pid in item_ids:
    sub   = str(product_id_to_title.get(pid, ''))
    main  = str(product_id_to_title_main.get(pid, ''))
    # size  = str(product_id_to_title_weight.get(pid, ''))
    text  = f"{sub}, {main}".strip()
    item_texts.append(text if text else "unknown product")

print(f"Товаров для эмбеддинга: {len(item_texts)}")
print("Примеры текстов:")
for t in item_texts[:5]:
    print(" •", t)

Товаров для эмбеддинга: 7093
Примеры текстов:
 • REFRIGERATED BISCUITS REGULAR, REFRGRATD DOUGH PRODUCTS
 • FLOUR: WHITE & SELF RISING, FLOUR & MEALS
 • CHICKEN DRUMS, CHICKEN
 • KIDS CEREAL, COLD CEREAL
 • REFRIGERATED COOKIES-BREAK N B, REFRGRATD DOUGH PRODUCTS


In [24]:
# print("Вычисление эмбеддингов")
F = sbert_model.encode(
    item_texts,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)
print(f"Матрица эмбеддингов F: {F.shape}")

Batches: 100%|██████████| 28/28 [00:10<00:00,  2.75it/s]


Матрица эмбеддингов F: (7093, 768)


In [25]:
product2id_val = product2id_train
id2product_val = id2product_train

val_grouped = val_df.groupby('basket_id').apply(
    lambda x: [p for p in x.product_id if p in product2id_val]
).reset_index()

val_grouped.rename(columns={0: 'interactions'}, inplace=True)
val_grouped = val_grouped[val_grouped['interactions'].apply(len) > 1].reset_index(drop=True)

print(f"val baskets size: {len(val_grouped)}")

val baskets size: 16671


In [26]:
X_matrix = sps.csr_matrix((np.ones(filtered_train.shape[0]), (filtered_train['basket_id'], filtered_train['product_id'])),
                          shape=(filtered_train['basket_id'].max() + 1, filtered_train['product_id'].max() + 1))

In [27]:
pre_complementary = pd.read_csv("complementary_pairs_v4.csv")
not_complementary = pd.read_csv("non_complementary_pairs_v4.csv")

pre_complementary_train = pd.read_csv("complementary_pairs_train.csv")
not_complementary_train = pd.read_csv("non_complementary_pairs_train.csv")

Logic_matrix = np.zeros(shape=(F.shape[0], F.shape[0]))
for index, row in pre_complementary.iterrows():
    if (row['id_1'] >= Logic_matrix.shape[0] or row['id_2'] >= Logic_matrix.shape[0]):
        continue
    Logic_matrix[row['id_1'], row['id_2']] = 1
    Logic_matrix[row['id_2'], row['id_1']] = 1
for index, row in not_complementary.iterrows():
    if (row['id_1'] >= Logic_matrix.shape[0] or row['id_2'] >= Logic_matrix.shape[0]):
        continue
    Logic_matrix[row['id_1'], row['id_2']] = -1
    Logic_matrix[row['id_2'], row['id_1']] = -1

for index, row in pre_complementary_train.iterrows():
    if (row['id1'] not in product2id_train.keys() or row['id2'] not in product2id_train.keys()):
        continue
    Logic_matrix[product2id_train[row['id1']], product2id_train[row['id2']]] = 1
    Logic_matrix[product2id_train[row['id2']], product2id_train[row['id1']]] = 1

for index, row in not_complementary_train.iterrows():
    if (row['id1'] not in product2id_train.keys() or row['id2'] not in product2id_train.keys()):
        continue
    Logic_matrix[product2id_train[row['id1']], product2id_train[row['id2']]] = -1
    Logic_matrix[product2id_train[row['id2']], product2id_train[row['id1']]] = -1

print(Logic_matrix.sum())

165238.0


In [28]:
def fit_l3ae(X_sparse, F, lambda_F=1.0, lambda_X=100.0, lambda_KD=10.0):
    X = X_sparse.toarray()
    X_t = torch.tensor(X, dtype=torch.float32, device=device)
    F_t = torch.tensor(F.T, dtype=torch.float32, device=device)
    n_items = X_t.size(1)

    G_F = F_t.T @ F_t + lambda_F * torch.eye(F_t.size(1), device=device)
    P_F = torch.linalg.inv(G_F)
    
    S = torch.eye(n_items, device=device) - P_F @ torch.diag(1.0 / torch.diag(P_F))

    XtX = X_t.T @ X_t
    reg_total = lambda_KD + lambda_X
    P_KD = torch.linalg.inv(XtX + reg_total * torch.eye(n_items, device=device))

    M_num = torch.eye(n_items, device=device) + lambda_KD * P_KD @ S
    mu = torch.diag(M_num) / torch.diag(P_KD)
    B = M_num - P_KD * mu.unsqueeze(0)
    B.fill_diagonal_(0)
    return B.cpu().numpy(), S.cpu().numpy()

In [38]:
def fit_l3ae_meaningful(X_sparse, Logic, lambda_F=1.0, lambda_X=100.0, lambda_KD=10.0):
    X = X_sparse.toarray()
    X_t = torch.tensor(X, dtype=torch.float32, device=device)
    Logic_t = torch.tensor(Logic, dtype=torch.float32, device=device)
    n_items = X_t.size(1)

    # S из Logic
    G_F = Logic_t + lambda_F * torch.eye(Logic_t.size(1), device=device)
    P_F = torch.linalg.inv(G_F)
    S = torch.eye(n_items, device=device) - P_F @ torch.diag(1.0 / torch.diag(P_F))

    XtX = X_t.T @ X_t
    reg_total = lambda_KD + lambda_X
    P_KD = torch.linalg.inv(XtX + reg_total * torch.eye(n_items, device=device))

    M_num = torch.eye(n_items, device=device) + lambda_KD * P_KD @ S
    mu = torch.diag(M_num) / torch.diag(P_KD)
    B = M_num - P_KD * mu.unsqueeze(0)
    B.fill_diagonal_(0)
    return B.cpu().numpy(), S.cpu().numpy()

In [30]:
def predict_(history_items, B, item2id, id2item, topn=10):
    encoded = [item2id[it] for it in history_items if it in item2id]
    if not encoded:
        return []
    user_vec = np.zeros(B.shape[0])
    user_vec[encoded] = 1.0

    scores = user_vec @ B

    scores[encoded] = -np.inf

    top_indices = np.argsort(scores)[-topn:][::-1]
    top_items = [id2item[i] for i in top_indices if scores[i] > -np.inf]
    return top_items

In [31]:
def evaluate_(val_grouped, X_matrix, model, **kwargs):
    if model == "l3ae":
        B, _ = fit_l3ae(X_sparse=X_matrix, F=kwargs.get('F'),
                        lambda_F=kwargs.get('lambda_F'),
                        lambda_X=kwargs.get('lambda_X'),
                        lambda_KD=kwargs.get('lambda_KD'))
    elif model == "ease":
        B = fit_ease(X_matrix, reg=kwargs.get('reg'))
    else:
        B, _ = fit_l3ae_meaningful(X_sparse=X_matrix, Logic=kwargs.get('Logic'),
                                   lambda_F=kwargs.get('lambda_F'),
                                   lambda_X=kwargs.get('lambda_X'),
                                   lambda_KD=kwargs.get('lambda_KD'))
    np.fill_diagonal(B, 0.0)

    topk = kwargs.get('topk')
    n_items = B.shape[0]
    history_list = []
    target_list = []

    # --- average-over-all: каждый айтем в корзине по очереди становится таргетом ---
    for _, row in val_grouped.iterrows():
        items = row['interactions']
        known = [it for it in items if it in product2id_train]
        if len(known) < 2:
            continue
        for i, target in enumerate(known):
            history = [it for j, it in enumerate(known) if j != i]
            target_enc = product2id_train[target]
            history_enc = [product2id_train[it] for it in history]
            target_list.append(target_enc)
            history_list.append(history_enc)

    n_valid = len(history_list)
    if n_valid == 0:
        return {'model': model, 'recall': 0, 'ndcg': 0, **kwargs}

    history_dense = np.zeros((n_valid, n_items), dtype=np.float32)
    for i, hist in enumerate(history_list):
        history_dense[i, hist] = 1.0

    B_t = torch.tensor(B, dtype=torch.float32, device=device)
    history_t = torch.tensor(history_dense, dtype=torch.float32, device=device)

    scores_t = history_t @ B_t

    for i, hist in enumerate(history_list):
        if hist:
            scores_t[i, hist] = -1e9

    _, topk_indices = torch.topk(scores_t, k=topk, dim=1)
    topk_indices = topk_indices.cpu().numpy()
    target_arr = np.array(target_list)

    recall = 0
    ndcg_sum = 0.0
    for i in range(n_valid):
        target = target_arr[i]
        recs = topk_indices[i]
        if target in recs:
            recall += 1
        y_true = [1 if idx == target else 0 for idx in recs]
        if sum(y_true) > 0:
            ideal_order = list(range(topk, 0, -1))
            ndcg_sum += ndcg_score([y_true], [ideal_order])

    return {
        'model': model,
        'recall': recall / n_valid,
        'ndcg': ndcg_sum / n_valid,
        **kwargs
    }

In [32]:
val_sample = val_grouped.sample(500, random_state=322).reset_index(drop=True)

In [ ]:
lambdaX_candidates = np.logspace(-1, 2, 15)
lambdaF_candidates = np.logspace(-1, 2, 15)
lambdaKD_candidates = np.logspace(-1, 2, 15)
results_l3ae = []
R, N = [], []
cnt = 0
for lf in lambdaF_candidates:
    for lx in lambdaX_candidates:
        for lkd in lambdaKD_candidates:
            # print(lf, lx, lkd)
            cnt += 1
            res = evaluate_(val_sample, X_matrix, "l3ae", F=F, lambda_F=lf, lambda_X=lx, lambda_KD=lkd, topk=10)
            results_l3ae.append(res)
            R.append(res['recall'])
            N.append(res['ndcg'])
            if (cnt % 100 == 0):
                print(cnt, max(R), max(N))
    # print(f"lambda_F={lf}: Recall@10={res['recall']:.4f}, NDCG@10={res['ndcg']:.4f}")

100 0.10862186014935506 0.062283576993231074
200 0.11518443086671193 0.06703700033525781
300 0.11518443086671193 0.0672065165529453
400 0.11518443086671193 0.0672065165529453
500 0.11586331749264539 0.0674616541277198
600 0.11586331749264539 0.0674616541277198
700 0.11631590857660104 0.06770003805607064
800 0.11631590857660104 0.06770003805607064
900 0.11699479520253452 0.0679376354855105
1000 0.11699479520253452 0.0679376354855105
1100 0.11699479520253452 0.06840668171749317
1200 0.11699479520253452 0.06840668171749317
1300 0.11699479520253452 0.06840668171749317
1400 0.11744738628649015 0.06840668171749317
1500 0.11880515953835709 0.0689578772668188
1600 0.11880515953835709 0.0689578772668188
1700 0.11880515953835709 0.06896697278090447
1800 0.11880515953835709 0.06900660092718724
1900 0.11880515953835709 0.06900660092718724
2000 0.11903145508033491 0.06911478629404372
2100 0.11903145508033491 0.06911478629404372
2200 0.11903145508033491 0.0696663265052722
2300 0.11903145508033491 0.

In [37]:
print("end")

end


In [43]:
# L3AE OUR IDEAS
lambdaX_candidates = np.logspace(0.3, 3, 10)
lambdaF_candidates = np.logspace(-1, 3, 10)
lambdaKD_candidates = np.logspace(0, 1.5, 15)
results_our = []
R, N = [], []
cnt = 0
for lx in lambdaX_candidates:
    for lkd in lambdaKD_candidates:
        for lf in lambdaF_candidates:
            # print(lx, lkd, lf)
            cnt += 1
            if (cnt % 100 == 0):
                print(cnt, max(R), max(N))
            res = evaluate_(val_sample, X_matrix, "our", Logic=Logic_matrix, lambda_F=lf, lambda_X=lx, lambda_KD=lkd, topk=10)
            R.append(res['recall'])
            N.append(res['ndcg'])
            results_our.append(res)
    # print(f"lambda_F={lf}: Recall@10={res['recall']:.4f}, NDCG@10={res['ndcg']:.4f}")

100 0.0982122652183752 0.05839012285745756
200 0.10771667798144377 0.0636007553493947
300 0.10771667798144377 0.06364603204337382
400 0.10771667798144377 0.06364603204337382
500 0.10930074677528853 0.06420449772190298
600 0.10975333785924417 0.06442682045225795
700 0.10975333785924417 0.06456108268937878
800 0.109979633401222 0.06535428143808379
900 0.109979633401222 0.06535428143808379
1000 0.109979633401222 0.06535428143808379
1100 0.109979633401222 0.06535428143808379
1200 0.109979633401222 0.06535428143808379
1300 0.109979633401222 0.06535428143808379
1400 0.109979633401222 0.06535428143808379
1500 0.109979633401222 0.06535428143808379


In [44]:
print("end")

end


In [40]:
reg_candidates = np.logspace(-1, 3, 15)

ease_results = []
for reg in reg_candidates:
    print("Now:", reg)
    res = evaluate_(val_sample, X_matrix, "ease", reg=reg, topk=10)
    ease_results.append(res)
    # print(res)
    print(f"reg={reg}: Recall@10={res['recall']:.4f}, NDCG@10={res['ndcg']:.4f}")

Now: 0.1
reg=0.1: Recall@10=0.0516, NDCG@10=0.0280
Now: 0.193069772888325
reg=0.193069772888325: Recall@10=0.0570, NDCG@10=0.0307
Now: 0.372759372031494
reg=0.372759372031494: Recall@10=0.0616, NDCG@10=0.0329
Now: 0.7196856730011519
reg=0.7196856730011519: Recall@10=0.0652, NDCG@10=0.0362
Now: 1.3894954943731375
reg=1.3894954943731375: Recall@10=0.0738, NDCG@10=0.0416
Now: 2.6826957952797246
reg=2.6826957952797246: Recall@10=0.0821, NDCG@10=0.0473
Now: 5.17947467923121
reg=5.17947467923121: Recall@10=0.0914, NDCG@10=0.0538
Now: 10.0
reg=10.0: Recall@10=0.0978, NDCG@10=0.0577
Now: 19.306977288832496
reg=19.306977288832496: Recall@10=0.1059, NDCG@10=0.0625
Now: 37.27593720314938
reg=37.27593720314938: Recall@10=0.1093, NDCG@10=0.0643
Now: 71.96856730011514
reg=71.96856730011514: Recall@10=0.1088, NDCG@10=0.0651
Now: 138.94954943731375
reg=138.94954943731375: Recall@10=0.1050, NDCG@10=0.0623
Now: 268.26957952797244
reg=268.26957952797244: Recall@10=0.1021, NDCG@10=0.0593
Now: 517.94746792

In [37]:
# L3AE OUR IDEAS

max_ndcg, max_recall = 0, 0
best_ndcg_our, best_recall_our = None, None
print("OUR GAME:")
for i in results_our:
    # print(i)
    if (i['recall'] > max_recall):
        best_recall_our = i
        max_recall = max(max_recall, i['recall'])
    if (i['ndcg'] > max_ndcg):
        best_ndcg_our = i
        max_ndcg = max(max_ndcg, i['ndcg'])
    
print("best for recall", best_recall_our)
print("RECALL:", max_recall)
print("best for ndcg", best_ndcg_our)
print("NDCG:", max_ndcg)

OUR GAME:
best for recall {'model': 'our', 'recall': 0.11676849966055669, 'ndcg': 0.06747982297370599, 'Logic': array([[ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       ...,
       [ 0.,  0.,  0., ...,  0., -1.,  0.],
       [ 0.,  0.,  0., ..., -1.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.]]), 'lambda_F': 88.58667904100822, 'lambda_X': 19.306977288832496, 'lambda_KD': 31.622776601683793, 'topk': 10}
RECALL: 0.11676849966055669
best for ndcg {'model': 'our', 'recall': 0.11563702195066757, 'ndcg': 0.06847781265974417, 'Logic': array([[ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       ...,
       [ 0.,  0.,  0., ...,  0., -1.,  0.],
       [ 0.,  0.,  0., ..., -1.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.]]), 'lambda_F': 12.742749857031335, 'lambda_X': 71.96856730011514, 'lambda_KD': 19.306977288832496, 

In [38]:
max_ndcg, max_recall = 0, 0
best_ndcg_l3ae, best_recall_l3ae = None, None
for i in results_l3ae:
    # print(i)
    if (i['recall'] > max_recall):
        best_recall_l3ae = i
        max_recall = max(max_recall, i['recall'])
    if (i['ndcg'] > max_ndcg):
        best_ndcg_l3ae = i
        max_ndcg = max(max_ndcg, i['ndcg'])
print("l3ae")
print("best for recall", best_recall_l3ae)
print("best for ndcg", best_ndcg_l3ae)

l3ae
best for recall {'model': 'l3ae', 'recall': 0.11925775062231274, 'ndcg': 0.06940445927666752, 'F': array([[-0.003182  , -0.08536544,  0.02027727, ..., -0.00909046,
        -0.05491344, -0.00431926],
       [-0.04085928, -0.04020944,  0.00109676, ..., -0.02109563,
        -0.03118032, -0.01298462],
       [ 0.0504622 ,  0.01073228,  0.00955424, ...,  0.0283979 ,
        -0.01783688, -0.03449559],
       ...,
       [-0.00240503,  0.00108718,  0.01200331, ...,  0.02342349,
         0.00634961, -0.02970573],
       [ 0.01299042, -0.03419974,  0.01121378, ...,  0.00992199,
        -0.00527837, -0.03732279],
       [-0.05529369, -0.06523355,  0.05233682, ...,  0.00557318,
        -0.05879167,  0.00589415]], dtype=float32), 'lambda_F': 13.894954943731374, 'lambda_X': 1.9306977288832496, 'lambda_KD': 61.054022965853264, 'topk': 10}
best for ndcg {'model': 'l3ae', 'recall': 0.11857886399637928, 'ndcg': 0.0696663265052722, 'F': array([[-0.003182  , -0.08536544,  0.02027727, ..., -0.0090904

In [63]:
max_ncdg, max_recall = 0, 0

best = None
for i in results_:
    # print(i['ndcg'], i['recall'])
    if (i['recall'] > max_recall):
        best = i
        max_recall = max(max_recall, i['recall'])
    # max_recall = max(max_recall, i['recall'])
print("best for ndcg:", best)

for i in ease_results:
    print(i)

best for ndcg: {'lambda_X': np.float64(0.1), 'lambda_KD': np.float64(10.0), 'recall': np.float64(0.09), 'ndcg': np.float64(0.050648915411141374)}


NameError: name 'ease_results' is not defined

In [47]:
B_l3ae, _ = fit_l3ae(
        X_sparse=X_matrix,
        F=F,
        lambda_F=0.1,
        lambda_X=0.1,
        lambda_KD=10
)
B_ease = fit_ease(X_matrix, reg=10)
B_our, _ = fit_l3ae_meaningful(
        X_sparse=X_matrix,
        Logic=Logic_matrix,
        lambda_F=0.001,
        lambda_X=25.118864315095795,
        lambda_KD=10.0
    )

Start!
Finish!
Start!
Finish!


In [53]:
random_products = np.random.choice(list(product2id_train.keys()), 50, replace=False)

**попросил нейронку сделать красивый вывод, чтобы не как для EASE**

In [55]:

topn = 5
print("=" * 70)
print(f"{'Товар':<30} {'EASE':>5}  vs  {'L3AE':>5} vs {'OUR':>5}")
print("=" * 70)


for item in random_products[:50]:
    name = product_id_to_title.get(item, '?')
    cat  = product_id_to_title_main.get(item, '?')
    enc  = product2id_train.get(item)
    if enc is None:
        continue

    # EASE: похожие по весам
    ease_weights = np.array(B_ease[enc]).ravel()
    top_ease = np.argpartition(ease_weights, -topn)[-topn:]
    top_ease = top_ease[np.argsort(ease_weights[top_ease])[::-1]]

    # L3AE: похожие по B_l3ae
    l3ae_weights = B_l3ae[enc]
    top_l3ae = np.argpartition(l3ae_weights, -topn)[-topn:]
    top_l3ae = top_l3ae[np.argsort(l3ae_weights[top_l3ae])[::-1]]

    # L3AE_OUR: похожие по нашему методу
    our_weights = B_our[enc]
    top_our = np.argpartition(our_weights, -topn)[-topn:]
    top_our = top_our[np.argsort(our_weights[top_our])[::-1]]
    
    print(f"\n {name} [{cat}]")
    print(f"{'EASE':<30} | {'L3AE':<30} | {'OUR':<30}")
    print(f"{'-'*30}-+-{'-'*30}-+-{'-'*30}")
    for a, b, c in zip(top_ease, top_l3ae, top_our):
        ea = product_id_to_title.get(id2product_train.get(a), '?')
        la = product_id_to_title.get(id2product_train.get(b), '?')
        our = product_id_to_title.get(id2product_train.get(c), '?')
        print(f"{ea:<30} | {la:<30} | {our:<30}")

Товар                           EASE  vs   L3AE vs   OUR

 BREADINGS  COATINGS CRUMBS [BAKING NEEDS]
EASE                           | L3AE                           | OUR                           
-------------------------------+--------------------------------+-------------------------------
REFRIGERATED COFFEE CREAMERS   | BREADINGS  COATINGS CRUMBS     | BREADINGS  COATINGS CRUMBS    
SHREDDED CHEESE                | BREADINGS  COATINGS CRUMBS     | BREADINGS  COATINGS CRUMBS    
ALL FAMILY CEREAL              | BREADINGS  COATINGS CRUMBS     | BREADINGS  COATINGS CRUMBS    
FACIAL TISSUE & PAPER HANDKE   | BREADINGS  COATINGS CRUMBS     | BREADINGS  COATINGS CRUMBS    
CHEESE: NATURAL BULK           | BREADINGS  COATINGS CRUMBS     | BREADINGS  COATINGS CRUMBS    

 PRESERVES JAM MARMALADE [PNT BTR/JELLY/JAMS]
EASE                           | L3AE                           | OUR                           
-------------------------------+--------------------------------+-----------